# Notebook 05
* Author: Melissa Ormeño
* Date: May 2024


Aims:
* Obtiene las mutaciones para enfermedades raras desde orphanet

In [1]:
#Orphanet data
import pandas as pd
import xml.etree.ElementTree as ET

# Cargar y parsear el archivo XML
tree = ET.parse('en_product6 (4).xml')
root = tree.getroot()

# Definir una función para extraer información de cada desorden
def parse_disorder(disorder):
    disorder_data = {}
    disorder_data['DisorderID'] = disorder.attrib.get('id')
    disorder_data['OrphaCode'] = disorder.find('OrphaCode').text if disorder.find('OrphaCode') is not None else None
    disorder_data['ExpertLink'] = disorder.find('ExpertLink').text if disorder.find('ExpertLink') is not None else None
    disorder_data['Name'] = disorder.find('Name').text if disorder.find('Name') is not None else None
    disorder_data['DisorderType'] = disorder.find('DisorderType/Name').text if disorder.find('DisorderType/Name') is not None else None
    disorder_data['DisorderGroup'] = disorder.find('DisorderGroup/Name').text if disorder.find('DisorderGroup/Name') is not None else None

    # Extraer información genética
    gene_info = disorder.find('DisorderGeneAssociationList/DisorderGeneAssociation/Gene')
    if gene_info is not None:
        disorder_data['GeneName'] = gene_info.find('Name').text if gene_info.find('Name') is not None else None
        disorder_data['GeneSymbol'] = gene_info.find('Symbol').text if gene_info.find('Symbol') is not None else None
        disorder_data['GeneLocus'] = gene_info.find('LocusList/Locus/GeneLocus').text if gene_info.find('LocusList/Locus/GeneLocus') is not None else None
    else:
        disorder_data['GeneName'] = None
        disorder_data['GeneSymbol'] = None
        disorder_data['GeneLocus'] = None

    return disorder_data

# Extraer todos los desórdenes en una lista
disorders = []
for disorder in root.find('DisorderList'):
    disorder_data = parse_disorder(disorder)
    disorders.append(disorder_data)

# Convertir la lista de desórdenes en un DataFrame
df = pd.DataFrame(disorders)

# Mostrar el DataFrame
df


,DisorderID,OrphaCode,ExpertLink,Name,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus
0,17601,166024,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Multiple epiphyseal dysplasia-macrocephaly-fac...,Disease,Disorder,kinesin family member 7,KIF7,15q26.1
1,5,93,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Aspartylglucosaminuria,Disease,Disorder,aspartylglucosaminidase,AGA,4q34.3
2,17604,166035,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Brachydactyly-short stature-retinitis pigmento...,Malformation syndrome,Disorder,CWC27 spliceosome associated cyclophilin,CWC27,5q12.3
3,6,585,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Multiple sulfatase deficiency,Disease,Disorder,sulfatase modifying factor 1,SUMF1,3p26.1
4,7,118,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Beta-mannosidosis,Disease,Disorder,mannosidase beta,MANBA,4q24
...,...,...,...,...,...,...,...,...,...
3956,31418,619948,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Early-onset autoimmunity-autoinflammation-immu...,Disease,Disorder,suppressor of cytokine signaling 1,SOCS1,16p13.13
3957,31423,619979,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Developmental delay-immunodeficiency-leukoence...,Disease,Disorder,NFE2 like bZIP transcription factor 2,NFE2L2,2q31.2
3958,31422,619972,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,CADINS disease,Disease,Disorder,caspase recruitment domain family member 11,CARD11,7p22.2
3959,31415,619363,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,NOCARH syndrome,Disease,Disorder,cell division cycle 42,CDC42,1p36.12


In [2]:
df['GeneSymbol'].nunique()

2598

In [3]:
#Data de genes TKL
# Datos con la lista de genes quinasa a trabajar 
#interpro_code = 'IPR000333'
family_name = 'STKR'
df_tkl = pd.read_csv('../data/Cancer/df_icgc_{0}.csv'.format(family_name))

In [4]:
lista_tkl_genes = list(df_tkl['Gen_names'].unique().tolist())
print(len(lista_tkl_genes))
print(sorted(lista_tkl_genes))

12
['ACVR1', 'ACVR1B', 'ACVR1C', 'ACVR2A', 'ACVR2B', 'ACVRL1', 'AMHR2', 'BMPR1A', 'BMPR1B', 'BMPR2', 'TGFBR1', 'TGFBR2']


In [5]:
df[df['GeneSymbol'].str.contains('ACVR1')]


,DisorderID,OrphaCode,ExpertLink,Name,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus
108,205,337,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Fibrodysplasia ossificans progressiva,Disease,Disorder,activin A receptor type 1,ACVR1,2q24.1


In [6]:
df_orphanet = df[df['GeneSymbol'].isin(lista_tkl_genes)]

In [7]:
df_orphanet['GeneSymbol'].unique()

array(['ACVR1', 'ACVRL1', 'BMPR1A', 'BMPR1B', 'BMPR2', 'TGFBR1'],
      dtype=object)

In [8]:
df_orphanet['Name'].unique()

array(['Fibrodysplasia ossificans progressiva',
       'Hereditary hemorrhagic telangiectasia',
       'Hereditary mixed polyposis syndrome',
       'Acromesomelic dysplasia, Grebe type',
       'Fibular aplasia-complex brachydactyly syndrome',
       'Drug- or toxin-induced pulmonary arterial hypertension',
       'Generalized juvenile polyposis/juvenile polyposis coli',
       'Marfan syndrome type 2', 'Familial colorectal cancer Type X',
       'Pulmonary venoocclusive disease',
       'Multiple self-healing squamous epithelioma',
       'Brachydactyly type A1', 'Brachydactyly type C'], dtype=object)

In [9]:
df_orphanet

,DisorderID,OrphaCode,ExpertLink,Name,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus
108,205,337,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Fibrodysplasia ossificans progressiva,Disease,Disorder,activin A receptor type 1,ACVR1,2q24.1
126,236,774,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Hereditary hemorrhagic telangiectasia,Disease,Disorder,activin A receptor like type 1,ACVRL1,12q13.13
634,17145,157794,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Hereditary mixed polyposis syndrome,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2
735,2010,2098,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,"Acromesomelic dysplasia, Grebe type",Malformation syndrome,Disorder,bone morphogenetic protein receptor type 1B,BMPR1B,4q22.3
1020,2413,2639,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Fibular aplasia-complex brachydactyly syndrome,Malformation syndrome,Disorder,bone morphogenetic protein receptor type 1B,BMPR1B,4q22.3
1108,20329,275786,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Drug- or toxin-induced pulmonary arterial hype...,Clinical group,Group of disorders,bone morphogenetic protein receptor type 2,BMPR2,2q33.1-q33.2
1557,21951,329971,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Generalized juvenile polyposis/juvenile polypo...,Clinical subtype,Subtype of disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2
1873,20629,284973,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Marfan syndrome type 2,Clinical subtype,Subtype of disorder,transforming growth factor beta receptor 1,TGFBR1,9q22.33
2327,23409,440437,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Familial colorectal cancer Type X,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2
2568,9801,31837,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Pulmonary venoocclusive disease,Disease,Disorder,bone morphogenetic protein receptor type 2,BMPR2,2q33.1-q33.2


In [10]:
df_orphanet['Name'].nunique()

13

In [11]:
tgfbr1_disease=df_orphanet[df_orphanet['GeneSymbol']=='BMPR1A']
print(tgfbr1_disease['Name'].unique())
tgfbr1_disease

['Hereditary mixed polyposis syndrome'
 'Generalized juvenile polyposis/juvenile polyposis coli'
 'Familial colorectal cancer Type X']


,DisorderID,OrphaCode,ExpertLink,Name,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus
634,17145,157794,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Hereditary mixed polyposis syndrome,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2
1557,21951,329971,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Generalized juvenile polyposis/juvenile polypo...,Clinical subtype,Subtype of disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2
2327,23409,440437,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Familial colorectal cancer Type X,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2


In [12]:
# Data de decipher ya esta filtrada or los genes a tranajar, este filtro se realiza en el notebook 05.Decipher.ipynb
#interpro_code = 'IPR000333'
family_name = 'STKR'
df_decipher = pd.read_csv('../data/mutaciones/decipher_mutations_{0}'.format(family_name))
df_decipher.head(2)

,gene symbol,gene mim,disease name,disease mim,confidence category,allelic requirement,mutation consequence,phenotypes,organ specificity list,pmids,panel,prev symbols,hgnc id,gene disease pair entry date,cross cutting modifier,mutation consequence flag,confidence value flag,comments,variant consequence,disease ontology
0,ACVR2B,602730,HETEROTAXY SYNDROME,207574,limited,monoallelic_autosomal,altered gene product structure,NaN,GI tract;Heart/Cardiovasculature/Lymphatic,NaN,DD,NaN,174,2015-07-22 16:14:34,NaN,NaN,Requires clinical review,NaN,missense_variant;inframe_deletion;inframe_inse...,NaN
1,TGFBR1,190181,LOEYS-DIETZ SYNDROME,609192,definitive,monoallelic_autosomal,altered gene product structure,HP:0000278;HP:0000347;HP:0000175;HP:0000766;HP...,Face;Skeleton;Heart/Cardiovasculature/Lymphatic,18070134;16791849;15731757;16928994;16596670,DD,MSSE;ESS1,11772,2015-07-22 16:15:11,NaN,restricted repertoire of mutations,NaN,NaN,gain_of_function_variant,NaN


In [13]:
df_decipher[df_decipher['gene symbol']=='BMPR1A']

,gene symbol,gene mim,disease name,disease mim,confidence category,allelic requirement,mutation consequence,phenotypes,organ specificity list,pmids,panel,prev symbols,hgnc id,gene disease pair entry date,cross cutting modifier,mutation consequence flag,confidence value flag,comments,variant consequence,disease ontology


In [14]:
import pandas as pd


# Renombrar columnas para que coincidan
df_orphanet.rename(columns={'GeneSymbol': 'GeneSymbol', 'Name': 'DiseaseName'}, inplace=True)
df_decipher.rename(columns={'gene symbol': 'GeneSymbol', 'disease name': 'DiseaseName'}, inplace=True)

# Concatenar los DataFrames verticalmente
combined_df = pd.concat([df_orphanet, df_decipher], axis=0, ignore_index=True)

# Llenar NaN en la columna 'Disease' combinada con valores de 'DiseaseName' si existen
combined_df['Disease'] = combined_df['DiseaseName'].fillna('')

# Eliminar la columna 'DiseaseName' si ya no es necesaria
combined_df.drop(columns=['DiseaseName'], inplace=True)


C:\Users\melif\AppData\Local\Temp\ipykernel_22104\3861784583.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_orphanet.rename(columns={'GeneSymbol': 'GeneSymbol', 'Name': 'DiseaseName'}, inplace=True)


In [15]:
combined_df.head(5)

,DisorderID,OrphaCode,ExpertLink,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus,gene mim,disease mim,...,prev symbols,hgnc id,gene disease pair entry date,cross cutting modifier,mutation consequence flag,confidence value flag,comments,variant consequence,disease ontology,Disease
0,205,337,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Disease,Disorder,activin A receptor type 1,ACVR1,2q24.1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fibrodysplasia ossificans progressiva
1,236,774,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Disease,Disorder,activin A receptor like type 1,ACVRL1,12q13.13,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hereditary hemorrhagic telangiectasia
2,17145,157794,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hereditary mixed polyposis syndrome
3,2010,2098,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Malformation syndrome,Disorder,bone morphogenetic protein receptor type 1B,BMPR1B,4q22.3,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Acromesomelic dysplasia, Grebe type"
4,2413,2639,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Malformation syndrome,Disorder,bone morphogenetic protein receptor type 1B,BMPR1B,4q22.3,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fibular aplasia-complex brachydactyly syndrome


In [16]:
tgfbr1_disease=combined_df[combined_df['GeneSymbol']=='BMPR1A']
print(tgfbr1_disease['Disease'].unique())
tgfbr1_disease

['Hereditary mixed polyposis syndrome'
 'Generalized juvenile polyposis/juvenile polyposis coli'
 'Familial colorectal cancer Type X']


,DisorderID,OrphaCode,ExpertLink,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus,gene mim,disease mim,...,prev symbols,hgnc id,gene disease pair entry date,cross cutting modifier,mutation consequence flag,confidence value flag,comments,variant consequence,disease ontology,Disease
2,17145,157794,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hereditary mixed polyposis syndrome
6,21951,329971,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Clinical subtype,Subtype of disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Generalized juvenile polyposis/juvenile polypo...
8,23409,440437,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Familial colorectal cancer Type X


In [17]:
df_clinvar = pd.read_csv('../data/mutaciones/clinvar_mutations')

In [18]:
df_clinvar.head(2)

,Protein change,Molecular consequence,VariationID,Condition(s),Variant type,Gene(s),Germline classification,Germline review status
0,R258M,missense variant,376364,Gastric adenocarcinoma,single nucleotide variant,ACVR1,Likely pathogenic,no assertion criteria provided
1,R258M,missense variant,376364,Brainstem glioma,single nucleotide variant,ACVR1,Likely pathogenic,no assertion criteria provided


In [19]:
df_clinvar[df_clinvar['Condition(s)']=='Familial colorectal cancer Type X']

,Protein change,Molecular consequence,VariationID,Condition(s),Variant type,Gene(s),Germline classification,Germline review status


In [20]:
# -merge de clinvar y combined_df por la columnas de genes
combined_df = pd.merge(df_clinvar, combined_df, left_on='Gene(s)', right_on='GeneSymbol')

# Filtrar las variaciones que están relacionadas con enfermedades raras
# filtrar por las columnas 'GeneSymbol', 'Disease', 'Protein change', 'Molecular consequence', 'VariationID','Germline classification'
rare_disease_variations = combined_df[['GeneSymbol', 'Disease', 'Protein change', 'Molecular consequence', 'VariationID','Germline classification']]


In [21]:
rare_disease_variations

,GeneSymbol,Disease,Protein change,Molecular consequence,VariationID,Germline classification
0,ACVR1,Fibrodysplasia ossificans progressiva,R258M,missense variant,376364,Likely pathogenic
1,ACVR1,FIBRODYSPLASIA OSSIFICANS PROGRESSIVA,R258M,missense variant,376364,Likely pathogenic
2,ACVR1,Fibrodysplasia ossificans progressiva,R258M,missense variant,376364,Likely pathogenic
3,ACVR1,FIBRODYSPLASIA OSSIFICANS PROGRESSIVA,R258M,missense variant,376364,Likely pathogenic
4,ACVR1,Fibrodysplasia ossificans progressiva,G328V,missense variant,376363,Likely pathogenic
...,...,...,...,...,...,...
5255,TGFBR2,Loeys-Dietz syndrome 2,E406Q,missense variant,12503,Pathogenic
5256,TGFBR2,Loeys-Dietz syndrome 2,E527Q,missense variant,12503,Pathogenic
5257,TGFBR2,Loeys-Dietz syndrome 2,E535Q,missense variant,12503,Pathogenic
5258,TGFBR2,Loeys-Dietz syndrome 2,A535fs,frameshift variant,12501,Pathogenic


In [22]:
rare_disease_variations['Disease'].nunique()

18

In [23]:
rare_disease_variations['Protein change'].nunique()

2315

In [24]:
rare_disease_variations['Disease'].unique()

array(['Fibrodysplasia ossificans progressiva',
       'FIBRODYSPLASIA OSSIFICANS PROGRESSIVA', 'HETEROTAXY SYNDROME',
       'Hereditary hemorrhagic telangiectasia',
       'Hereditary mixed polyposis syndrome',
       'Generalized juvenile polyposis/juvenile polyposis coli',
       'Familial colorectal cancer Type X',
       'Acromesomelic dysplasia, Grebe type',
       'Fibular aplasia-complex brachydactyly syndrome',
       'Brachydactyly type A1', 'Brachydactyly type C',
       'BRACHYDACTYLY TYPE A2',
       'Drug- or toxin-induced pulmonary arterial hypertension',
       'Pulmonary venoocclusive disease', 'Marfan syndrome type 2',
       'Multiple self-healing squamous epithelioma',
       'LOEYS-DIETZ SYNDROME', 'Loeys-Dietz syndrome 2'], dtype=object)

In [25]:
rare_disease_variations[rare_disease_variations['GeneSymbol']=='BMPR1A']

,GeneSymbol,Disease,Protein change,Molecular consequence,VariationID,Germline classification
747,BMPR1A,Hereditary mixed polyposis syndrome,Y192fs,frameshift variant|non-coding transcript variant,3236417,Likely pathogenic
748,BMPR1A,Generalized juvenile polyposis/juvenile polypo...,Y192fs,frameshift variant|non-coding transcript variant,3236417,Likely pathogenic
749,BMPR1A,Familial colorectal cancer Type X,Y192fs,frameshift variant|non-coding transcript variant,3236417,Likely pathogenic
750,BMPR1A,Hereditary mixed polyposis syndrome,Y278fs,frameshift variant|non-coding transcript variant,3236417,Likely pathogenic
751,BMPR1A,Generalized juvenile polyposis/juvenile polypo...,Y278fs,frameshift variant|non-coding transcript variant,3236417,Likely pathogenic
...,...,...,...,...,...,...
2104,BMPR1A,Generalized juvenile polyposis/juvenile polypo...,W271*,nonsense,8231,Pathogenic
2105,BMPR1A,Familial colorectal cancer Type X,W271*,nonsense,8231,Pathogenic
2106,BMPR1A,Hereditary mixed polyposis syndrome,Q239*,nonsense,8230,Pathogenic
2107,BMPR1A,Generalized juvenile polyposis/juvenile polypo...,Q239*,nonsense,8230,Pathogenic


In [26]:
GEN=rare_disease_variations[rare_disease_variations['GeneSymbol']=='TGFBR2']

In [27]:
GEN['Protein change'].nunique()

845

In [28]:
import pandas as pd

# Crear un diccionario con los DataFrames por gen
genes = ['ACVR1', 'ACVR1B', 'ACVR1C', 'ACVR2A', 'ACVR2B', 'BMPR1A', 'BMPR1B', 'BMPR2', 'TGFBR1', 'TGFBR2', 'AMHR2', 'ACVRL1']
gene_dfs = {gene: rare_disease_variations[rare_disease_variations['GeneSymbol'] == gene] for gene in genes}


# Función para detallar mutaciones por gen
def detallar_mutaciones(df):
    return df.groupby(['GeneSymbol', 'Disease', 'Molecular consequence'])['Protein change'].apply(list).reset_index(name='mutaciones')

# Crear diccionarios para almacenar los resultados
detalles_por_gen = {}

# Aplicar las funciones a cada DataFrame en el diccionario
for gene, df in gene_dfs.items():
    detalles_por_gen[gene] = detallar_mutaciones(df)


In [29]:
print("Detalle de mutaciones:")
detalles_por_gen['BMPR1A']

Detalle de mutaciones:


,GeneSymbol,Disease,Molecular consequence,mutaciones
0,BMPR1A,Familial colorectal cancer Type X,frameshift variant,"[F147fs, I122fs, T391fs, Q117fs, G65fs, S113fs..."
1,BMPR1A,Familial colorectal cancer Type X,frameshift variant|5 prime UTR variant|non-cod...,[D27fs]
2,BMPR1A,Familial colorectal cancer Type X,frameshift variant|5 prime UTR variant|non-cod...,[R9fs]
3,BMPR1A,Familial colorectal cancer Type X,frameshift variant|non-coding transcript variant,"[Y192fs, Y278fs, Y304fs, Y306fs, Y322fs, Y331f..."
4,BMPR1A,Familial colorectal cancer Type X,frameshift variant|non-coding transcript varia...,"[S171fs, S143fs, V162fs, V134fs, V134fs, V162f..."
5,BMPR1A,Familial colorectal cancer Type X,inframe_deletion|initiator_codon_variant,[M1del]
6,BMPR1A,Familial colorectal cancer Type X,missense variant,"[C100Y, C72Y, G71R, G99R, H232D, H318D, H344D,..."
7,BMPR1A,Familial colorectal cancer Type X,missense variant|initiator_codon_variant,"[M1I, M1I, M1L, M1L, M1I, M1I, M1V, M1V]"
8,BMPR1A,Familial colorectal cancer Type X,nonsense,"[W274*, W221*, W247*, W265*, W135*, W249*, C10..."
9,BMPR1A,Familial colorectal cancer Type X,nonsense|5 prime UTR variant|non-coding transc...,[Y5*]


In [30]:
import pandas as pd

# Supongamos que 'rare_disease_variations' es tu DataFrame
# Cambiar el nombre de las filas en la columna 'Disease'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'Brachydactyly type A1', 'Disease'] = 'Brachydactyly type A1, A2, C'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'Brachydactyly type C', 'Disease'] = 'Brachydactyly type A1, A2, C'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'Brachydactyly type A2', 'Disease'] = 'Brachydactyly type A1, A2, C'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'BRACHYDACTYLY TYPE A2', 'Disease'] = 'Brachydactyly type A1, A2, C'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'LOEYS-DIETZ SYNDROME', 'Disease'] = 'Loeys-Dietz syndrome'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'FIBRODYSPLASIA OSSIFICANS PROGRESSIVA', 'Disease'] = 'Fibrodysplasia ossificans progressiva'


In [31]:
# crea una lista con los 18 tipos de cancer a tratar
lista_rd = rare_disease_variations['Disease'].unique()
#Crea un dataframe para cada tipo de grafico, para poder graficarlos de forma independiente
dict_rd = {}

# Iterar sobre cada tipo de cáncer en lista_cancer
for tipo in lista_rd:
    # Filtrar el DataFrame original para obtener solo las filas correspondientes al tipo de cáncer actual
    df_disease = rare_disease_variations[rare_disease_variations['Disease'] == tipo]
    
    # Asignar el DataFrame filtrado al diccionario con la clave correspondiente al tipo de cáncer
    dict_rd[tipo] = df_disease

# Ahora se puede acceder a cada DataFrame utilizando la clave correspondiente al tipo de cáncer en el diccionario



In [32]:
for Disease, data in sorted(dict_rd.items(), key=lambda x: len(x[1]['Protein change'].unique()), reverse=True):
    print(f"Número de mutaciones en enfermedades raras {Disease}: {len(data['Protein change'].unique())}")


Número de mutaciones en enfermedades raras Loeys-Dietz syndrome 2: 845
Número de mutaciones en enfermedades raras Hereditary hemorrhagic telangiectasia: 534
Número de mutaciones en enfermedades raras Hereditary mixed polyposis syndrome: 353
Número de mutaciones en enfermedades raras Generalized juvenile polyposis/juvenile polyposis coli: 353
Número de mutaciones en enfermedades raras Familial colorectal cancer Type X: 353
Número de mutaciones en enfermedades raras Drug- or toxin-induced pulmonary arterial hypertension: 330
Número de mutaciones en enfermedades raras Pulmonary venoocclusive disease: 330
Número de mutaciones en enfermedades raras Marfan syndrome type 2: 258
Número de mutaciones en enfermedades raras Multiple self-healing squamous epithelioma: 258
Número de mutaciones en enfermedades raras Loeys-Dietz syndrome: 258
Número de mutaciones en enfermedades raras Acromesomelic dysplasia, Grebe type: 22
Número de mutaciones en enfermedades raras Fibular aplasia-complex brachydact

In [33]:
for Disease, data in sorted(dict_rd.items(), key=lambda x: len(x[1]['Protein change'].unique()), reverse=True):
    print(f"Número de mutaciones en enfermedades raras {Disease}: {len(data['Protein change'].unique())}")


Número de mutaciones en enfermedades raras Loeys-Dietz syndrome 2: 845
Número de mutaciones en enfermedades raras Hereditary hemorrhagic telangiectasia: 534
Número de mutaciones en enfermedades raras Hereditary mixed polyposis syndrome: 353
Número de mutaciones en enfermedades raras Generalized juvenile polyposis/juvenile polyposis coli: 353
Número de mutaciones en enfermedades raras Familial colorectal cancer Type X: 353
Número de mutaciones en enfermedades raras Drug- or toxin-induced pulmonary arterial hypertension: 330
Número de mutaciones en enfermedades raras Pulmonary venoocclusive disease: 330
Número de mutaciones en enfermedades raras Marfan syndrome type 2: 258
Número de mutaciones en enfermedades raras Multiple self-healing squamous epithelioma: 258
Número de mutaciones en enfermedades raras Loeys-Dietz syndrome: 258
Número de mutaciones en enfermedades raras Acromesomelic dysplasia, Grebe type: 22
Número de mutaciones en enfermedades raras Fibular aplasia-complex brachydact

In [34]:
import altair as alt
alt.data_transformers.enable('default', max_rows=None)

g1=alt.Chart(rare_disease_variations).mark_bar().encode(
    x=alt.X('Disease',axis=alt.Axis(
              labelAngle=40, labelFontSize=12), sort='-y'),
    y=alt.Y('count(Protein change):Q').title('Numero de mutaciones'),
    color=alt.Color('GeneSymbol', scale=alt.Scale(scheme='category20'), title='Genes')
).properties(
    title='Numero de mutaciones en enfermedades raras'
)

g2=alt.Chart(rare_disease_variations).mark_arc().encode(
    theta='count(Protein change)):Q',
    color="Molecular consequence"
).properties(
    title='Tipo de mutacion mas frecuente en cancer'
)
g1

alt.Chart(...)

In [35]:
alt.Chart(rare_disease_variations).transform_aggregate(
    unique_aa_mutation='distinct(Protein change)',
    groupby=['GeneSymbol','Disease', 'Protein change']
).mark_bar().encode(
    y=alt.Y('count(unique_aa_mutation):Q', title='Número de mutaciones'),
    x=alt.X('Disease', sort='-y', title='Enfermedad rara',axis=alt.Axis(
              labelAngle=-40, labelFontSize=10)),
    color=alt.Color('GeneSymbol', scale=alt.Scale(scheme='category20'), title='Genes')
).properties(
    title="Número de mutaciones en enfermedades raras"
).properties(
        width=600,
        height=500
    )

alt.Chart(...)

In [36]:
# Guardar los resultados en un archivo CSV
# Datos con la lista de genes quinasa a trabajar 
#interpro_code = 'IPR000333'
family_name = 'STKR'
rare_disease_variations.to_csv('../data/mutaciones/rare_disease_variations_{0}.csv'.format(family_name), index=False)

In [37]:
rare_disease_variations

,GeneSymbol,Disease,Protein change,Molecular consequence,VariationID,Germline classification
0,ACVR1,Fibrodysplasia ossificans progressiva,R258M,missense variant,376364,Likely pathogenic
1,ACVR1,Fibrodysplasia ossificans progressiva,R258M,missense variant,376364,Likely pathogenic
2,ACVR1,Fibrodysplasia ossificans progressiva,R258M,missense variant,376364,Likely pathogenic
3,ACVR1,Fibrodysplasia ossificans progressiva,R258M,missense variant,376364,Likely pathogenic
4,ACVR1,Fibrodysplasia ossificans progressiva,G328V,missense variant,376363,Likely pathogenic
...,...,...,...,...,...,...
5255,TGFBR2,Loeys-Dietz syndrome 2,E406Q,missense variant,12503,Pathogenic
5256,TGFBR2,Loeys-Dietz syndrome 2,E527Q,missense variant,12503,Pathogenic
5257,TGFBR2,Loeys-Dietz syndrome 2,E535Q,missense variant,12503,Pathogenic
5258,TGFBR2,Loeys-Dietz syndrome 2,A535fs,frameshift variant,12501,Pathogenic
